# Synthesis: every run in one folder

Run this once the generation runs are done. It collects every condition into one Drive folder with a single index of all images and, where judge results exist, the recognition statistics and paired comparisons. **CPU runtime is enough.**

| Condition | Run folder (default) | Made by |
|---|---|---|
| `clip_strokes` | `runs/prod150_v1/images` | prod150 (done) |
| `clipag_strokes` | `runs/clipag150_v1/images` | `clipag_cohort.ipynb` |
| `clip_strokes_control` | `runs/clip150ctl_v1/images` | `clipag_cohort.ipynb`, control15 |
| `clip_pixels` | `runs/clip150px_v1/images` | `clipag_pixels.ipynb` |
| `clipag_pixels` | `runs/clipag150px_v1/images` | `clipag_pixels.ipynb` |

Conditions whose folder doesn't exist yet are skipped, so you can run this early to check progress. Nothing in the run folders is changed: everything is copied.

**What it writes** to `ExperimentResults/synthesis/<OUTPUT_NAME>/`:
- `images/<condition>/<word>.png` and `metadata/<condition>/<word>.json`: every image and its record
- `index.csv`: one row per image (condition, word, imageability, generation time and loss, judge results)
- `wide.csv`: one row per word, conditions side by side
- `grids/words/<group>/<word>.png`: one word across all conditions; `grids/overview_<group>.png`: contact sheets
- `judge_input/`: every image in judge-pipeline layout, to judge all conditions in one run
- `SUMMARY.md`, `summary.json`: completeness, top-1 by imageability group, paired comparisons

**Two passes.** First pass: collect and look. Then judge the new runs (see section 4). Second pass: add the new judge `combined.csv` files to `JUDGE_CSVS` and run again with a new `OUTPUT_NAME` to get the statistics.

In [ ]:
# @title 1. Settings
PROJECT_FOLDER = "MyDrive/ExperimentResults"  # @param {type:"string"}
OUTPUT_NAME = "synthesis_v1"  # @param {type:"string"}
SOURCE_REF = "experiment/clipdraw-clipag"  # @param {type:"string"}
REPO_URL = "https://github.com/ArmitaAfr/Visibility_Thresholds.git"

# Condition name -> (folder relative to PROJECT_FOLDER, guidance model, substrate).
CONDITIONS = {
    "clip_strokes":         ("runs/prod150_v1/images",    "clip",   "strokes"),
    "clipag_strokes":       ("runs/clipag150_v1/images",  "clipag", "strokes"),
    "clip_strokes_control": ("runs/clip150ctl_v1/images", "clip",   "strokes"),
    "clip_pixels":          ("runs/clip150px_v1/images",  "clip",   "pixels"),
    "clipag_pixels":        ("runs/clipag150px_v1/images","clipag", "pixels"),
}

# Judge outputs (analysis/combined.csv), relative to PROJECT_FOLDER. Missing files are skipped.
# Images are matched by PNG SHA-256, so any evaluation run can be listed. Earlier files win.
JUDGE_CSVS = [
    "runs/prod150_eval9f_v1/stages/balanced150/analysis/combined.csv",
    "runs/clipag150_eval_v1/stages/balanced150/analysis/combined.csv",
    "runs/synthesis_eval_v1/stages/balanced150/analysis/combined.csv",
]

# Paired comparisons (baseline, treatment). Pairs with a missing condition are skipped.
COMPARISONS = [
    ("clip_strokes", "clipag_strokes"),        # main question: CLIPAG vs CLIP, strokes
    ("clip_strokes", "clip_strokes_control"),  # code-path check (15 words)
    ("clip_strokes", "clip_pixels"),           # strokes vs pixels, CLIP
    ("clipag_strokes", "clipag_pixels"),       # strokes vs pixels, CLIPAG
    ("clip_pixels", "clipag_pixels"),          # CLIPAG vs CLIP, pixels
]

## 2. Drive, source and word list

In [ ]:
import subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = (Path("/content/drive") / PROJECT_FOLDER).resolve()
if not DRIVE_ROOT.is_dir():
    raise ValueError(f"Project folder not found on Drive: {DRIVE_ROOT}")

SRC = Path("/content/Visibility_Thresholds")
if not SRC.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(SRC)], check=True)
subprocess.run(["git", "fetch", "origin", SOURCE_REF], cwd=SRC, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=SRC, check=True)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SRC, text=True).strip())

sys.path.insert(0, str(SRC))
import importlib
import clipag_cohort.cohort as cohort
import clipag_cohort.compare as compare
import clipag_cohort.synthesize as synthesize
for m in (cohort, compare, synthesize):
    importlib.reload(m)

subprocess.run(["git", "fetch", "origin", cohort.SAMPLED_WORDS_COMMIT], cwd=SRC, check=True)
WORDS = cohort.load_words(subprocess.check_output(
    ["git", "show", f"{cohort.SAMPLED_WORDS_COMMIT}:{cohort.SAMPLED_WORDS_PATH}"], cwd=SRC))

## 3. Collect, check and write

In [ ]:
import pandas as pd
from IPython.display import Markdown, Image as ShowImage, display

conditions = {}
for name, (folder, guidance, substrate) in CONDITIONS.items():
    root = DRIVE_ROOT / folder
    if root.is_dir():
        conditions[name] = {"root": root, "guidance": guidance, "substrate": substrate}
        print(f"found    {name:22} {folder}")
    else:
        print(f"skipped  {name:22} {folder} (not found)")
if not conditions:
    raise ValueError("No condition folders found. Check PROJECT_FOLDER and CONDITIONS.")

judge_tables = []
for rel in JUDGE_CSVS:
    path = DRIVE_ROOT / rel
    if path.is_file():
        judge_tables.append((rel, pd.read_csv(path)))
        print(f"judges   {rel}")
    else:
        print(f"no file  {rel}")

pairs = [(b, t) for b, t in COMPARISONS if b in conditions and t in conditions]
OUT = DRIVE_ROOT / "synthesis" / OUTPUT_NAME
index, summary = synthesize.synthesize(conditions, WORDS, OUT, judge_tables, comparisons=pairs)
print(f"\nWrote {len(index)} images to {OUT}")
display(Markdown((OUT / "SUMMARY.md").read_text()))

In [ ]:
# Contact sheets: words down, conditions across
for group in ("low", "medium", "high"):
    sheet = OUT / "grids" / f"overview_{group}.png"
    if sheet.is_file():
        print(group)
        display(ShowImage(filename=str(sheet), width=900))

## 4. Judging the new runs

Statistics need judge results for each condition. Use the team's judge notebook (`visibility_thresholds_colab_benchv1.ipynb`) with a new evaluation `RUN_ID`, `SOURCE_LINEAGE = None`, and either:

- **All conditions at once:** `EXISTING_ROOTS = [str(DRIVE_ROOT / "synthesis/<OUTPUT_NAME>/judge_input")]`, RUN_ID `synthesis_eval_v1`. The judge's `part` column will be the condition name. If that judge version refuses several images of the same word in one run, use the next option instead.
- **One run at a time:** `EXISTING_ROOTS = [str(DRIVE_ROOT / "runs/clipag150_v1/images")]` with its own evaluation RUN_ID, and so on per run.

Then `MANIFEST` → `GPU_EVALUATE` → `CPU_EVALUATE`. Add each new `analysis/combined.csv` to `JUDGE_CSVS` above and rerun this notebook with a new `OUTPUT_NAME`. Already-judged images (prod150) don't need judging again.